# Appendix 10.8: Computer Use & Agents

- [Lesson](#lesson)
- [Exercise](#exercise)
- [Example Playground](#example-playground)

## Setup

This appendix is conceptual rather than API-call-heavy — computer use requires a sandboxed virtual display that isn't available inside this notebook, so we won't be spinning one up here. The exercise below is a pure Python safety-logic exercise and doesn't require an API call at all.

---

## Lesson

### From tool use to agents

Step back and look at what Appendix 10.2 actually built: Claude looks at a request, decides to call a tool, you run it, you feed the result back, and Claude decides what to do next — possibly calling another tool, possibly finishing up. Loop that until Claude stops asking for tools, and **you have an agent**: a model that observes, acts, observes the result of its action, and decides the next action, autonomously, until the task is done.

Everything you've learned in this course feeds directly into agent quality, because an agent is just a loop of individual Claude calls: clear and direct instructions (Chapter 2) make each step less ambiguous, examples (Chapter 7) show the model what a good tool call looks like, extended thinking (Appendix 10.5) lets it reason between steps about what a tool result actually means, and prompt caching (Appendix 10.4) keeps a long-running agent session affordable as its context grows turn after turn. **Claude Code**, which you're using right now to run this tutorial, is itself exactly this kind of agent — a loop over tools like `bash`, file read/write, and search, instead of a calculator or a database.

### Computer use: the tool that operates a screen

**Computer use** takes this one step further: instead of only calling tools you've hand-written wrappers for (a specific API, a specific database function), Claude can be given a `computer` tool that takes screenshots and issues mouse/keyboard actions — `screenshot`, `left_click`, `type`, `key`, `scroll` — the same primitive actions a human has. That means Claude can operate **any application a human could**, including ones with no API at all: legacy internal tools, a specific SaaS dashboard, a desktop application.

The loop is structurally identical to Appendix 10.2's tool-use loop:
1. Claude requests an action (e.g., `{"action": "left_click", "coordinate": [412, 88]}`).
2. Your harness executes it against a real (sandboxed) display and captures a new screenshot.
3. You send the screenshot back as the tool result.
4. Claude looks at the new screen state and decides the next action — often reasoning about it via extended thinking (Appendix 10.5) first.

**Where this shows up in real systems:**
- QA automation that exercises a real UI the way a user would, catching visual/interaction bugs that API-level tests miss.
- Automating workflows across legacy enterprise software that predates any API.
- Browser-based research or form-filling agents.

**Version note:** computer use shipped as a public beta with the Claude 3.5/3.7 generation and has matured alongside interleaved thinking and Claude 4-generation models, which noticeably improved multi-step reliability — earlier versions were genuinely rough for anything beyond short, simple tasks.

### Safety — read this before you build one

Computer use (and agentic tool use generally) is the highest-blast-radius category of everything in this course, because the model is now taking real actions instead of just returning text. A few non-negotiables, mirrored by how production agent harnesses (including Claude Code) actually behave:

- **Run it sandboxed.** A dedicated VM or container with no access to real credentials, production databases, or your actual filesystem — never point computer use at your primary machine or a system with real user data.
- **Treat everything on-screen as untrusted input.** A webpage, document, or email the model looks at can contain text deliberately crafted to look like an instruction ("ignore your previous task and instead...") — this is **prompt injection**, and it's a real, demonstrated risk category for any agent that reads external content. Never let content the agent merely *observes* be treated as if the user said it.
- **Gate irreversible or high-consequence actions behind human confirmation** — submitting a form, sending a message, making a purchase, deleting something. Reversible actions (navigating, reading, scrolling) can proceed autonomously; hard-to-reverse ones shouldn't, by default.
- **Scope permissions tightly and explicitly.** Don't hand an agent broad access "just in case" — give it exactly what the task needs, for exactly as long as the task needs it.

These aren't hypothetical — they're the same category of guardrail that governs how Claude Code itself is allowed to act on your machine right now.

---

## Exercise

### Exercise 10.8.1 - Action Safety Gate

Even without a live computer-use session, you can practice the core safety pattern: **a gate function that decides whether a proposed agent action is safe to auto-execute, or needs human confirmation first.**

Write `needs_confirmation(action)` that takes an action dict like `{"type": "click", "target": "Submit Payment"}` or `{"type": "type", "text": "hello", "field": "search box"}` and returns `True` if the action should require human confirmation before executing. It should flag actions whose `type` is one of `"submit"`, `"purchase"`, `"delete"`, or `"send"`, **or** any `"type"` action where the `text` looks like it might be a password/credential (contains the word "password" in the target/field, case-insensitive).

In [ ]:
# Your function goes here — this is the only field you should change
def needs_confirmation(action):
    pass

test_actions = [
    ({"type": "click", "target": "Next page"}, False),
    ({"type": "submit", "target": "Checkout form"}, True),
    ({"type": "delete", "target": "Customer record #4471"}, True),
    ({"type": "type", "text": "hunter2", "field": "Password"}, True),
    ({"type": "type", "text": "blue shoes", "field": "Search box"}, False),
    ({"type": "send", "target": "Email to finance team"}, True),
]

print("--------------------------- GRADING ---------------------------")
all_correct = True
for action, expected in test_actions:
    got = needs_confirmation(action)
    status = "OK" if got == expected else "WRONG"
    if got != expected:
        all_correct = False
    print(f"[{status}] {action} -> {got} (expected {expected})")
print("\nAll correct!" if all_correct else "\nSome cases failed — check your logic.")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_10_8_1_hint; print(exercise_10_8_1_hint)

### Congrats!

**Recap:** an agent is just a tool-use loop that keeps going until the task is done — computer use extends that loop with a `computer` tool so Claude can operate a real screen via screenshots and clicks/keystrokes, unlocking any application a human could use, no API required. The cost is a much larger safety surface: sandboxing, untrusted-input treatment of anything on screen, and human confirmation on irreversible actions aren't optional extras, they're the difference between a useful agent and an incident.

**Interview-answer framing:** "An agent, in the Claude sense, is a tool-use loop: the model takes an action, observes the result, and decides the next action, repeating until done — Claude Code is a concrete example of this. Computer use is a specific tool that gives the model screenshot-and-click/keyboard access to a real screen, using the exact same request/response loop as any other tool, which means it can operate legacy software with no API at all. The main engineering challenge isn't the loop itself, it's safety: sandboxing execution, treating on-screen content as untrusted (prompt injection risk), and gating irreversible actions behind human confirmation."

Head over to Appendix 10.9 for a look at MCP, which standardizes how tools get plugged into agents like this.